In [ ]:
# Install required packages
!pip install -q zarr segmentation-models-pytorch

In [ ]:
from torch.utils.data import Dataset,DataLoader
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import WeightedRandomSampler
import torch.nn.functional as F
import cv2
import torch
import os
import zarr
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm
from pathlib import Path
import torch.nn as nn
import copy
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from skimage.morphology import skeletonize
import seaborn as sns
import json
import matplotlib.patches as mpatches
import datetime
import shutil
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from albumentations.core.transforms_interface import ImageOnlyTransform
from torchvision.models.swin_transformer import (
swin_t,swin_s,swin_b,
swin_v2_t,swin_v2_s,swin_v2_b,
Swin_S_Weights,Swin_V2_S_Weights,
Swin_B_Weights,Swin_V2_B_Weights,
Swin_T_Weights,Swin_V2_T_Weights
)
from transformers import AutoModelForSemanticSegmentation
import random
from sklearn.metrics import accuracy_score

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import PolynomialLR
from torchvision.transforms import v2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm.notebook import tqdm
import json
import cv2
import matplotlib.pyplot as plt
import numpy as np

# dataset

In [ ]:
class UnetDataset(Dataset):
    def __init__(self,transform,data,base_size=512,valid=False):
        super(UnetDataset,self).__init__()
        self.data = data
        self.transform = transform
        self.base_size=base_size
        self.valid=valid
    def __len__(self):
        return len(self.data)
    def __getitem__(self,index):
        img, mask , rare_mask ,common_mask = self.data[index]
        # img = np.expand_dims(img, axis=-1) 
        mask = mask[...,None]
        rare_mask = rare_mask[...,None]
        common_mask = common_mask[...,None]

        result = self.transform(
            image=img,
            mask=mask,
            rare_mask = rare_mask,
            common_mask = common_mask,
        )

        new_image = result['image']
        new_mask = result["mask"].squeeze(-1)
        new_common_mask = result["common_mask"].squeeze(-1)
        new_rare_mask = result["rare_mask"].squeeze(-1)
    

        new_common_mask = new_common_mask.long()
        new_rare_mask = new_rare_mask.long()
        new_mask = new_mask.long()
        # new_image = self.to_tensor(image = new_image)["image"]
        return (
            new_image.float() ,
            new_common_mask.unsqueeze(0),
            new_rare_mask.unsqueeze(0),
            new_mask
        )

class ValidUnetDataset(Dataset):
    def __init__(self,transform,data):
        super(ValidUnetDataset,self).__init__()
        self.data = data
        self.transform = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self,index):
        img,side_label,binary_mask,abs_mask,mask = self.data[index]
        # img = np.expand_dims(img, axis=-1) 
        mask = mask[...,None]
        binary_mask = binary_mask[...,None]
        abs_mask = abs_mask[...,None]

        result = self.transform(
            image=img,
            mask=mask,
            binary_mask = binary_mask,
            abs_mask = abs_mask
        )
        new_image = result['image']
        new_mask = result['mask'].squeeze(-1)
        new_abs_mask = result["abs_mask"].squeeze(-1)
        new_binary_mask = result["binary_mask"].squeeze(-1)

        new_binary_mask = new_binary_mask.long()
        new_abs_mask = new_abs_mask.long()
        new_mask = new_mask.long()

        # new_image = self.to_tensor(image = new_image)["image"]
        return new_image.float() , side_label ,new_binary_mask , new_abs_mask , new_mask

class UnetExampleDataset(Dataset):
    def __init__(self,transform,data,base_transform=None):
        super(UnetExampleDataset,self).__init__()
        
        self.data = data
        self.transform = transform
        self.to_tensor = ToTensorV2()
        if(base_transform is None):
            self.base_transform = A.Compose(
                [ToTensorV2()]
            )
        else:
            self.base_transform = base_transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self,index):
        img,side_label,binary_mask,abs_mask,mask = self.data[index]
        # img = np.expand_dims(img, axis=-1) 
        # print(img.shape)
        mask = mask[...,None]
        binary_mask = binary_mask[...,None]
        abs_mask = abs_mask[...,None]
        
        result = self.transform(
            image=img,
            mask=mask,
            binary_mask = binary_mask,
            abs_mask = abs_mask
        )
        new_image = result['image']
        new_mask = result['mask'].squeeze(-1)
        new_abs_mask = result["abs_mask"].squeeze(-1)
        new_binary_mask = result["binary_mask"].squeeze(-1)
        
        raw_result = self.base_transform(
            image=img,
            mask=mask,
            binary_mask = binary_mask,
            abs_mask = abs_mask
        )
        raw_image = raw_result['image']
        raw_mask = raw_result['mask'].squeeze(-1)
        raw_abs_mask = raw_result["abs_mask"].squeeze(-1)
        raw_binary_mask = raw_result["binary_mask"].squeeze(-1)
        new_image = self.to_tensor(image = new_image)["image"]
        return new_image.float() , new_mask  , raw_image.float() , raw_mask
    
def make_dataloader(data,args,valid=False,sampler_weights=None):
    if(sampler_weights is not None):
        print("using weighted sampler here")
        sampler = WeightedRandomSampler(sampler_weights, len(sampler_weights))
        dataloader = DataLoader(
            data,
            batch_size = args["batch_size"] ,
            num_workers = args["num_workers"] ,
            pin_memory=True,
            shuffle=False,
            sampler=sampler
        )
        
    else : 
        if(valid):
            print("valid with no sampler")
            dataloader = DataLoader(
                data,
                batch_size = args["batch_size"] ,
                num_workers = args["num_workers"] ,
                pin_memory=True,
                shuffle=False,
            )
        else : 
            print("train with no sampler")
            dataloader = DataLoader(
                data,
                batch_size = args["batch_size"] ,
                num_workers = args["num_workers"] ,
                pin_memory=True,
                shuffle=True
            )
    return dataloader

# helpers

In [ ]:
@torch.no_grad()
def read_images(base_path, part,preprocessor,in_c,resize_binary,rare,
                max_workers=None,train_class_counts=None,k=40):
    base_path = Path(base_path)
    images_base = base_path / "images" / part
    labels_base = base_path / "labels" / part
    skels_base = base_path / "skels" / part
    with open(f"data/{part}.json","r") as f:
        side_labels = json.load(f)
    if(train_class_counts is not None):
        freqs = train_class_counts / train_class_counts.sum()
        class_w = 1 / (freqs + 1e-6)
        class_w[0] = 0
    image_names = sorted([p.name for p in os.scandir(images_base) if p.is_file()])
    if(train_class_counts is not None):
        max_count = np.max(train_class_counts[1:])
        print("max count is : ",max_count)
    if(not preprocessor):
        print("NOTE : preprocessor is not defined . no preprocessing will be used !")
    def _read_one(fname):
        name_stem = Path(fname).stem
        img_path = images_base / fname
        label_path = labels_base / f"{name_stem}.zarr"
        skel_path = skels_base / fname
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        
        if(preprocessor):
            img = preprocessor(img)
        if(in_c!=1):
            img = cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)

        label = zarr.load(str(label_path))
        if(train_class_counts is not None):
            num_classes = class_w.shape[0]
            counts = np.bincount(label.reshape(-1), minlength=num_classes)
            weight = float((counts * class_w).sum())
        else:
            weight = None

        # NOTE : REMOVE LATER 
        common_label = np.zeros_like(label)
        rare_label = np.zeros_like(label)

        unique_labels = np.unique(label)
        unique_labels = unique_labels[unique_labels!=0]
        for ul in unique_labels:
            if(ul in rare):
                rare_label[label==ul] = 1
            else:
                common_label[label==ul] = 1

        side_label = side_labels[name_stem]
        return img, label , rare_label ,common_label , weight

    if max_workers is None:
        cpu = os.cpu_count() or 4
        max_workers = min(32, cpu * 4)

    results = []
    weights = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for img, label , rare_label ,common_label , weight in tqdm(ex.map(_read_one, image_names), total=len(image_names)):
            results.append([img, label , rare_label ,common_label])
            weights.append(weight)

    if(train_class_counts is not None):
        weights = torch.tensor(weights).double()
        weights +=1e-8
        weights /=weights.sum()
        return results , weights
    else:
        return results

def to_device(img,gt_mask,device,binary_mode):
    gt_mask = gt_mask.long()
    img = img.to(device)
    gt_mask = gt_mask.to(device)
    if(binary_mode):
        gt_label = gt_label.to(device)
    else :
        gt_label = None
    return img , gt_mask 


def crop_dims(target , current):
    left = (current.shape[3]-target.shape[3])//2
    right = (current.shape[3]-target.shape[3]) - left
    top = (current.shape[2]-target.shape[2])//2
    down = (current.shape[2]-target.shape[2]) - top
    croped = current[:,:,top:-down , left:-right]
    return croped
def padd_dims(target , current):
    pad_h = target.shape[2] - current.shape[2] 
    pad_w = target.shape[3] - current.shape[3]
    padded = F.pad(current, (0, pad_w, 0, pad_h), mode='constant', value=0)
    return padded

@torch.no_grad()
def TP_TN_FP_FN(preds,gt,process_preds=True,return_TN=False):
    if(process_preds):
        preds_argmax = torch.argmax(preds,dim=1)
        onehot_preds = F.one_hot(preds_argmax,num_classes=preds.shape[1])
        pred_onehot = onehot_preds.permute(0, 3, 1, 2).float()
    else :
        pred_onehot = preds
        
    onehot_gt = F.one_hot(gt,num_classes=preds.shape[1])
    onehot_gt = onehot_gt.permute(0, 3, 1, 2).float()
    TN = 0
    if(return_TN):
        TN = (((1-onehot_gt)*(1-pred_onehot)).sum(dim=(0,2,3))).cpu()
    TP = ((onehot_gt*pred_onehot).sum(dim=(0,2,3))).cpu()
    
    FP = (((1-onehot_gt)*pred_onehot).sum(dim=(0,2,3))).cpu()
    FN = ((onehot_gt*(1-pred_onehot)).sum(dim=(0,2,3))).cpu()
    return TP , TN , FP , FN

def to_rgb(img):
    if(isinstance(img,np.ndarray)):
        img = torch.from_numpy(img)
    x_disp = (img- img.min()) / (img.max() - img.min() + 1e-8)
    
    img = torch.cat([x_disp,x_disp,x_disp],dim=0) *255
    # print("d",img.shape)
    img = img.permute(1,2,0)
    return img.numpy()
def denorm(img,mean,std):
    if(isinstance(img,np.ndarray)):
        img = torch.from_numpy(img)
    img = (img*std) + mean
    img = img.clamp(0,1)*255
    return img.permute(1,2,0).cpu().numpy().astype(np.uint8)

def draw_mask(image,mask,args=None,colors=None):
    img = image.copy().astype(np.uint8)
    m = mask.astype(np.int64)
    if(colors is None):
        colors = np.array([(0,255,0)]*25,dtype=np.uint8)
        c = np.array([(0,255,0)])
        
    else:
        
        c = colors[m[m>0]-1].reshape(-1,3)
        
    # print("----")
    # print(c.shape)
    # print(img[m>0].shape)
    img[m>0] = c
    # print(img.shape)
    return img

@torch.no_grad()
def plot_some_images(data,transforms,mean,std,image_counts=36,fig_shape=(6,6),
                     base_transforms=None):
    ds = UnetExampleDataset(transform=transforms , data=data,base_transform=base_transforms)
    dataloader = DataLoader(
        ds,
        batch_size = 2 ,
        num_workers = 4 ,
        pin_memory=False,
        shuffle=True
    )

    iter_loader = iter(dataloader)
    w,h=fig_shape
    plt.figure(figsize=(w*5,h*5))
    for i in range(1,image_counts+1,2):
        new_imgs , new_mask , old_imgs , old_mask = next(iter_loader)
   
        new_img = new_imgs[0]
        old_img = old_imgs[0]
        if(new_img.shape[0]==1):
            new_img = to_rgb(new_img)
            old_img = to_rgb(old_img)

        else:
            new_img = denorm(new_img,mean,std)
            old_img = denorm(old_img,mean,std)

        # print(old_img.min(),old_img.max())
        # print(new_img.min(),new_img.max())
        # print("--------------")
        new_img = draw_mask(new_img,new_mask[0].numpy())
        
        # print(old_img.min(),old_img.max())
        plt.subplot(w,h,i)
        plt.imshow(old_img)
        plt.title("Old Image")

        plt.subplot(w,h,i+1)
        plt.imshow(new_img)
        plt.title("New Image")

def pre_hard_skeletonize(base_path,output_path):
    parts = ["train","val","test"]
    os.makedirs(os.path.join(output_path , "skels"),exist_ok=True)
    for part in parts:
        mask_base_path = os.path.join(base_path,"labels",part)
        os.makedirs(os.path.join(output_path , "skels",part),exist_ok=True)

        mask_list = os.listdir(mask_base_path)
        for mask_name in tqdm(mask_list):
            name = Path(mask_name).stem
            mask_path = os.path.join(mask_base_path,mask_name)

            mask = zarr.load(str(mask_path))
       
            mask = (mask!=0).astype(np.uint8)
        
            out_skel_path = os.path.join(output_path,"skels",part,f"{name}.png")
            skel = skeletonize(mask).astype(np.uint8) * 255
            cv2.imwrite(out_skel_path,skel)
@torch.no_grad()
def pre_soft_skeletonize(base_path,output_path,batch_size=10,k=25):
    parts = ["train","val","test"]
    os.makedirs(os.path.join(output_path , "skels_soft"),exist_ok=True)
    for part in parts:
        mask_base_path = os.path.join(base_path,"labels",part)
        os.makedirs(os.path.join(output_path , "skels_soft",part),exist_ok=True)

        mask_list = os.listdir(mask_base_path)
        mask_buffer = []
        name_buffer = []
        for i,mask_name in enumerate(tqdm(mask_list)):
            name = Path(mask_name).stem
            mask_path = os.path.join(mask_base_path,mask_name)
            mask = zarr.load(str(mask_path))
            mask = (mask!=0).astype(np.float32)

            mask = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0)
            mask_buffer.append(mask)
            name_buffer.append(name)
            if((i+1)%batch_size==0 or i==len(mask_list)-1):
                mask_buffer = torch.cat(mask_buffer,dim=0).to("cuda")
                skels = soft_skeletonize(mask_buffer,k=k)
                skels = skels.cpu().numpy()
                B = skels.shape[0]
                for i in range(B):
                    skel = skels[i,0].astype(np.uint8)*255
                    o_name = name_buffer[i]
                    out_skel_path = os.path.join(
                        output_path,"skels_soft",part,f"{o_name}.png")
                    cv2.imwrite(out_skel_path,skel)
                mask_buffer = []
                name_buffer = []

@torch.no_grad()
def compute_confution_matrix(data_loader,model,class_maps,output_folder_path=None,draw_plot = True,class_count=26,use_amp=False):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    conf_mat = torch.zeros((class_count,class_count))
    model.eval()
    for img,common_mask,rare_mask,masks in data_loader:
        img = img.to(device)
        with torch.autocast(device_type=device,dtype=torch.float16,enabled=use_amp):
            mask = masks.to(device).view(-1)
            pred_masks = model(img)[-1]

        pred_mask = torch.argmax(pred_masks,dim=1).view(-1)
        encoded_results = (mask*class_count + pred_mask).cpu()
        counts = torch.bincount(encoded_results,minlength=class_count**2).view(class_count,class_count)
        conf_mat += counts
        
    conf_mat = conf_mat.float() / conf_mat.sum(dim=1,keepdims=True).clamp(min=1)
    conf_mat = conf_mat.numpy()
    if(draw_plot):
        class_names = ["background" for i in range(class_count)]
        for index , name in class_maps.items():
            class_names[index] = name
        plt.figure(figsize=(20,20))
        ax = sns.heatmap(
            conf_mat,
            annot=True,
            fmt=".2f",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues"
        )
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        ax.set_title("Confusion Matrix")
        plt.tight_layout()
        if(output_folder_path):
            out_path = os.path.join(output_folder_path,"conf_mat.png")
            plt.savefig(out_path)
    return conf_mat

def erode(mask):
    h_pool = -F.max_pool2d(-mask,(3,1),(1,1),(1,0))
    v_pool = -F.max_pool2d(-mask,(1,3),(1,1),(0,1))
    return torch.min(v_pool,h_pool)
def dilate(mask):
    return F.max_pool2d(mask,(3,3),(1,1),(1,1))
def soft_open(mask):
    return dilate(erode(mask))
def soft_skeletonize(I,k=25):
    I_ = soft_open(I)
    S = F.relu(I-I_)
    for i in range(k):
        I = erode(I)
        I_ = soft_open(I)
        S = S + (1-S)*F.relu(I-I_)
    return S
def labels_to_string(mask,remove_bg = True,max_size =13):
    s = 0
    if(remove_bg):
        s=1
    u = np.unique(mask)[1:].tolist()
    u = list(map(str,u))
    c = ""
    for i,label in enumerate(u) : 
        c+=label
        if((i+1)%max_size==0):
            c+="\n"
        else:
            c+="|"
    return c

# logger

In [ ]:
colors = np.array([
    (242,  24,  24),   # Red
    (242,  77,  24),   # Red-Orange
    (242, 129,  24),   # Orange
    (242, 181,  24),   # Yellow-Orange
    ( 24, 242, 216),   # Cyan
    (242, 234,  24),   # Yellow
    (146,  24, 242),   # Purple
    (199, 242,  24),   # Yellow-Green
    (146, 242,  24),   # Lime
    ( 94, 242,  24),   # Green
    (242,  24, 181),   # Fuchsia
    ( 42, 242,  24),   # Green (brighter)
    ( 94,  24, 242),   # Violet
    ( 24, 242,  59),   # Spring Green
    (242,  24, 129),   # Pink
    ( 24, 242, 111),   # Aquamarine
    ( 24, 242, 164),   # Turquoise
    ( 24, 164, 242),   # Azure
    (199,  24, 242),   # Magenta
    ( 24, 216, 242),   # Sky Blue
    ( 24, 111, 242),   # Blue
    (242,  24, 234),   # Hot Pink
    ( 24,  59, 242),   # Royal Blue
    ( 42,  24, 242),   # Indigo
    (242,  24,  77),   # Rose
], dtype=np.uint8)


@torch.no_grad()
def save_full_report(recorder,output_base_path,model,valid_loader,binary_type,
                     args,class_map,mean,std,name=None,just_binary_trining=False,use_amp=False):
    now = datetime.datetime.now()
    save_folder_name = str(now)
    if(name):
        save_folder_name += f" [{name}]"
    output_folder_path = os.path.join(output_base_path,save_folder_name)

    os.makedirs(output_folder_path,exist_ok=True)
    
    print("Save Model")
    torch.save(model.state_dict(), os.path.join(output_folder_path,"model.pth"))

    print("Saving Memory")
    save_memory(recorder,args,output_folder_path)

    print("Saving All Plots")
    draw_loss_plots(recorder , output_folder_path)
    draw_avg_metric_plots(recorder , output_folder_path)
    draw_all_metric_plots(recorder , output_folder_path)
    if(not just_binary_trining):
        compute_confution_matrix(
            data_loader=valid_loader,
            model = model,
            class_maps = class_map,
            draw_plot = True,
            class_count=len(class_map)+1,
            output_folder_path=output_folder_path,
            use_amp = use_amp
            
        )
    print("Saving Examples")
    draw_examples(
        model=model,
        valid_loader=valid_loader,
        args=args,
        class_map=class_map,
        output_folder_path=output_folder_path,
        mean=mean,std=std,
        just_binary_trining=just_binary_trining,
        use_amp = use_amp,
        binary_type = binary_type
    )

    print("Saving Verbal Results")
    write_verbal_results(recorder,output_folder_path,just_binary_trining)

    print("Copying Notebook To Results")
    if(args["just_binary_trining"]):
        if(args["binary_type"]=="common"):

            notebook_name = "commen main.ipynb"
        elif(args["binary_type"]=="rare"):
            notebook_name = "rare main.ipynb"
        else:
            notebook_name = "nnUnetAttention.ipynb"
    else:
        notebook_name = "nnUnetAttention.ipynb"
    notebook_out_path = os.path.join(output_folder_path,"notebook.ipynb") 
    shutil.copyfile(f"./{notebook_name}",notebook_out_path )
    print("builfding kaggle project")
    build_kaggle_project(output_folder_path,notebook_name)

def write_verbal_results(recorder,output_base_path,just_binary_trining=False):
    report = ""
    report_path = os.path.join(output_base_path,"report.txt")
    losses_keys = recorder.losses_keys
    with open("./data/train_count.json","r") as f:
        train_count = json.load(f)
    for part,data in recorder.metric_avg_list.items():
        report +=f"======= > {part} verbal Report < =======\n"

        dice_list = data["dice"]
        precison_list = data["precision"]
        recall_list = data["recall"]

        best_idx = int(np.argmax(dice_list))
        
        best_dice = dice_list[best_idx]
        best_precision = precison_list[best_idx]
        best_recall = recall_list[best_idx]


        report += (
            f"best epoch : [{best_idx+1}]\n"
            f"best dice : [{best_dice}] - best precision : [{best_precision}] - best recall : [{best_recall}] \n"
        )
    
        for loss_name in losses_keys:
            loss_list = recorder.history[part][loss_name]
            best_loss = loss_list[best_idx]

            report += f"bset {loss_name} : [{best_loss}] - "
            
        report+="\n"
        if(not just_binary_trining):
            for index , c in recorder.class_maps.items():
                dice = recorder.metric_history[part]["dice"][index][best_idx]
                precision = recorder.metric_history[part]["precision"][index][best_idx]
                recall = recorder.metric_history[part]["recall"][index][best_idx]

                counts = train_count[c]
                report += f"{c} => dice : {dice} - p : {precision} - r : {recall} || train counts : {counts}\n"
        report +="<=><=><=><=><=><=><=><=><=><=><=><=><=><=><=><=><=>\n"
    with open(report_path , "w") as f : 
        f.write(report)

def save_memory(recorder,args,output_folder_path):
    history_path = os.path.join(output_folder_path,"loss_history.json")
    full_metric_path = os.path.join(output_folder_path,"full_metric_hostory.json")
    avg_metric_path = os.path.join(output_folder_path,"avg_metric_hostory.json")
    args_path = os.path.join(output_folder_path,"args.json")

    with open(history_path , "w") as f:
        json.dump(recorder.history,f,indent=4)
    with open(full_metric_path , "w") as f:
        json.dump(recorder.metric_history,f,indent=4)
    with open(avg_metric_path , "w") as f:
        json.dump(recorder.metric_avg_list,f,indent=4)
    with open(args_path , "w") as f:
        json.dump(args,f,indent=4)

def draw_loss_plots(recorder,output_folder_path):
    plt.figure(figsize=(15,20))
    
    losses_keys = recorder.losses_keys
    colors = ["g","r","b","y","orange"]
    colors_per_class = {}

    for i,loss_name in enumerate(losses_keys):
        colors_per_class[loss_name] = colors[i]

    plt_path =os.path.join(output_folder_path,"loss_plot.png")
    
    for i,part in enumerate(recorder.history):
        plt.subplot(2,1,i+1)
        for loss_name,data in recorder.history[part].items():
            length = len(data)-1
            x = np.arange(length)
            plt.plot(x,data[:-1],color = colors_per_class[loss_name],label=loss_name)
        plt.title(f"{part} loss plot")
        plt.legend()
    plt.savefig(plt_path,dpi=150)

def draw_avg_metric_plots(recorder,output_folder_path):
    plt.figure(figsize=(15,20))
    plt_path =os.path.join(output_folder_path,"avg_metrics.png")
    for i,part in enumerate(recorder.metric_avg_list):
        plt.subplot(2,1,i+1)

        dice_data = recorder.metric_avg_list[part]["dice"]
        precision_data = recorder.metric_avg_list[part]["precision"]
        recall_data = recorder.metric_avg_list[part]["recall"]

        length = len(dice_data)
        x = np.arange(length)
        plt.plot(x,dice_data,color="g",label="dice")
        plt.plot(x,precision_data,color="r",label="precision")
        plt.plot(x,recall_data,color="b",label="recall")
        plt.title(f"{part} avg dice plot")
        plt.legend()
    plt.savefig(plt_path)
def draw_all_metric_plots(recorder,output_folder_path):
    for part in recorder.history: 
        plt_path =os.path.join(output_folder_path,f"{part}_full_metric.png")
        plt.figure(figsize=(30,30))
        for i , class_index  in enumerate(recorder.metric_history[part]["dice"]):
            dice_data = recorder.metric_history[part]["dice"][class_index]
            precision_data = recorder.metric_history[part]["precision"][class_index]
            recall_data = recorder.metric_history[part]["recall"][class_index]

            class_name = recorder.class_maps[class_index]
            plt.subplot(5,5,i+1)
            length = len(dice_data)
            x = np.arange(length)
            plt.plot(x,dice_data,color="g",label="dice")
            plt.plot(x,precision_data,color="r",label="precision")
            plt.plot(x,recall_data,color="b",label="recall")
            plt.title(f"{class_name}")
            plt.legend()

        plt.savefig(plt_path)



@torch.no_grad()
def draw_examples(model,valid_loader,args,class_map,just_binary_trining,
                  output_folder_path,mean=None,std=None,w=6,h=6,
                  binary_type=None,use_amp=False):
    plt_path = os.path.join(output_folder_path,"examples.png")
    plt.figure(figsize=(30,30))
    plot_count =18
    patches = [
        mpatches.Patch(color=np.array(colors[j-1]) / 255.0, label=f"{j}:{class_map[j]}")
        for j in range(1,len(class_map)+1)
    ]
    i=0
    img_index=1
    valid_iterator = iter(valid_loader)
    model.eval()
    for i in range(plot_count):
        img,common_mask,rare_mask,mask= next(valid_iterator)
        with torch.autocast(device_type=args["device"],dtype=torch.float16,enabled=use_amp):
            pred_masks = model(img.to(args["device"]))[-1]
        if(just_binary_trining):
            if(binary_type=="common"):
                chosen_mask = common_mask.squeeze(1)
            else:
                chosen_mask = rare_mask.squeeze(1)
        else:
            chosen_mask = mask
        pred_mask = pred_masks[0].cpu().numpy()# 26 x H , W
        pred_mask = np.argmax(pred_mask,axis=0)# H , W

        mask = chosen_mask[0].numpy() # H,W
        img = img[0]# C , H , W

        if(img.shape[0]==1):
            img = to_rgb(img)
        else : 
            img = denorm(img,mean,std)
        real_annoted = draw_mask(
            img,
            mask,
            args,
            colors = colors if not just_binary_trining else None
        )
        pred_annoted = draw_mask(
            img,
            pred_mask,
            args,
            colors = colors if not just_binary_trining else None
        )
        
        gt_classes = labels_to_string(mask)
        pred_classes = labels_to_string(pred_mask)
        plt.subplot(h,w,img_index)
        plt.imshow(real_annoted)
        plt.title(f"Ground Truth:{gt_classes}")
        plt.subplot(h,w,img_index+1)
        plt.imshow(pred_annoted)
        plt.title(f"Predicted:{pred_classes}")
        img_index+=2
        if(img_index-1==h):
            plt.legend(
                handles=patches,
                bbox_to_anchor=(1.05, 1),
                loc='upper left',
                borderaxespad=0.,
                title="Classes"
            )
        i+=1
        
    plt.savefig(plt_path)

# preprocessing

In [ ]:
class CLAHE : 
    def __init__(self,clipLimit=2.0,tileGridSize=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)
    def __call__(self,img):
        enhanced = np.clip(img, 0, 255)
        return self.clahe.apply(enhanced)
class WhiteTopHat:
    def __init__(self,kernel_size = (50, 50)):
        self.kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_size)
    def __call__(self,img):
        neg_img = cv2.bitwise_not(img)
        tophat_img = cv2.morphologyEx(neg_img, cv2.MORPH_TOPHAT, self.kernel,borderType=cv2.BORDER_REPLICATE)
        return cv2.subtract(img, tophat_img)
class DoubleClahe:
    def __init__(self,kernel_size=(15, 15), clip_limit=2.0, tile_grid=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
        self.kernel = cv2.getStructuringElement(cv2.MORPH_RECT,kernel_size)
        
    def __call__(self,img):
        clah_1 = self.clahe.apply(img)
        img_inv = cv2.bitwise_not(clah_1)
        top_hat = cv2.morphologyEx(img_inv,cv2.MORPH_TOPHAT,self.kernel)
        img_sub = cv2.subtract(img_inv,top_hat)
        clah_2 = self.clahe.apply(img_sub)
        return clah_2
 
# Augementations 
def normalize_xca(img, **kwargs):
    x = img.astype(np.float32, copy=False)
    m = x > 0
    if np.any(m):
        mean = x[m].mean()
        std  = x[m].std()
        x[m] = (x[m] - mean) / (std + 1e-8)
        x[~m] = 0.0
    else:
        x = x / 1.0
    return x

def morph_binary_mask(x, **kwargs):
    m = x.copy()
    
    if m.shape[-1] == 3:
        
        m2 = cv2.cvtColor(m,cv2.COLOR_RGB2GRAY)
    else:
        m2 = m

    m2 = (m2 > 0).astype(np.uint8)

    if np.random.rand() < 0.5:
        k = np.ones((3, 3), np.uint8)
       
        if np.random.rand() < 0.5:
            m2 = cv2.dilate(m2, k, iterations=1)
        else:
            m2 = cv2.erode(m2, k, iterations=1)

    if np.random.rand() < 0.5:
        blurred = cv2.GaussianBlur(m2.astype(np.float32), (3, 3), 0)
        m2 = (blurred > 0.5).astype(np.uint8)

    if np.random.rand() < 0.5:
        h, w = m2.shape        
        for _ in range(200):
            y = np.random.randint(0, h)
            x = np.random.randint(0, w)
            m2[y, x] = 0

    
    if m.shape[-1] == 3:
        m_out = cv2.cvtColor(m[...,0],cv2.COLOR_GRAY2RGB)
    else:
        m_out = m2

    return m_out

def laplacian_pyramid_enhance(img, levels=3):

    img = img.astype(np.float32) / 255.0

    G = [img]
    for i in range(1, levels):
        G.append(cv2.pyrDown(G[i-1]))

    L = []
    for i in range(levels - 1):
        GE = cv2.pyrUp(G[i+1], dstsize=(G[i].shape[1], G[i].shape[0]))
        L.append(G[i] - GE)

    H = np.array([[0, -1,  0],
                  [-1, 5, -1],
                  [0, -1,  0]], dtype=np.float32)

    L_enh = [cv2.filter2D(Li, -1, H) for Li in L]

    current = G[-1]
    for i in reversed(range(levels - 1)):
        current = cv2.pyrUp(current, dstsize=(L_enh[i].shape[1], L_enh[i].shape[0]))
        current = current + L_enh[i]

    current = np.clip(current, 0, 1)
    return (current * 255).astype(np.uint8)

def gaussian_diffrential_scale_inverse(img,sigma_small=1.0,sigma_large=3.0,k=0.5):
    img_norm = (img).astype(np.uint8)/255.5
    mu = cv2.GaussianBlur(img_norm,(0,0),sigma_small)
    sq = cv2.GaussianBlur(img_norm**2 , (0,0),sigma_small)
    local_std = np.sqrt(np.maximum(sq - mu**2, 0)) + 1e-6

    lc = (img_norm - mu)/local_std

    lc_norm = (lc - lc.min())/(lc.max()-lc.min() + 1e-6)

    g_small = cv2.GaussianBlur(img_norm , (0,0),sigma_small)
    g_big = cv2.GaussianBlur(img_norm , (0,0),sigma_large)

    contrast_weights = np.clip(0.5 + k * (lc_norm - 0.5), 0, 1)
    enhanced = contrast_weights * g_small + (1 - contrast_weights) * g_big

    enhanced = np.clip(enhanced, 0, 1)
    return (enhanced*255).astype(np.uint8)


if __name__ == "__main__" :
    # img = cv2.imread("./dataset/syntax/images/train/10.png",cv2.IMREAD_GRAYSCALE)

    # trans1 = DoubleClahe(kernel_size=(3,3),tile_grid=(3,3))
    # trans2 = DoubleClahe(kernel_size=(3,3),tile_grid=(5,5)) 
    # trans3 = DoubleClahe(kernel_size=(3,3),tile_grid=(8,8))
    # trans4 = DoubleClahe(kernel_size=(3,3),tile_grid=(10,10))  

    # plt.figure(figsize=(10,10))
    # plt.subplot(3,3,1)
    # plt.imshow(cv2.cvtColor(trans1(img),cv2.COLOR_GRAY2RGB))
    # plt.subplot(3,3,2)
    # plt.imshow(cv2.cvtColor(trans2(img),cv2.COLOR_GRAY2RGB))
    # plt.subplot(3,3,3)
    # plt.imshow(cv2.cvtColor(trans3(img),cv2.COLOR_GRAY2RGB))
    # plt.subplot(3,3,4)
    # plt.imshow(cv2.cvtColor(trans4(img),cv2.COLOR_GRAY2RGB))
    # plt.subplot(3,3,5)
    # plt.imshow(cv2.cvtColor(img,cv2.COLOR_GRAY2RGB))
    # plt.show()
    print(morph_binary_mask(np.random.rand(448, 448, 3).astype(np.uint8)).shape)

# recorder

In [ ]:
class HistoryRecorder:
    def __init__(self,class_maps,losses_keys,class_count = 25):

        self.history = {
            "train":{},
            "valid":{}
        }
        for key in losses_keys:
            self.history["train"][key] = [[]]
            self.history["valid"][key] = [[]]
        self.losses_keys = losses_keys
        self.metric_history={}
        self.class_maps =class_maps
        for part in self.history :
            self.metric_history[part]={"dice":{},"precision":{},"recall":{}}
            for i in range(1,class_count):
                self.metric_history[part]["dice"][i]=[]
                self.metric_history[part]["recall"][i]=[]
                self.metric_history[part]["precision"][i]=[]
                
        self.metric_avg_list = {
            "train":{"dice":[],"precision":[],"recall":[]},
            "valid":{"dice":[],"precision":[],"recall":[]}
        }

        self.class_count = class_count 
        
    def add_losses(self,part,loss_dict):
        for loss_name,loss in loss_dict.items():
            self.history[part][loss_name][-1] += [loss]
        
    def add_metrics(self,dice,precision,recall,part):
        dice = dice[1:]
        precision = precision[1:]
        recall = recall[1:]
        for i in range(self.class_count-1):
            d = dice[i]
            r = recall[i]
            p = precision[i]
            self.metric_history[part]["dice"][i+1].append(d)
            self.metric_history[part]["recall"][i+1].append(r)
            self.metric_history[part]["precision"][i+1].append(p)
    def avg_losses(self,part):
        for key in self.history[part]:
            self.history[part][key][-1] = np.mean(self.history[part][key][-1])
            self.history[part][key].append([])
            
    def print_loss_report(self,part,epoch,avg_first=True):
        if(avg_first):
            self.avg_losses(part)
            
        report = f"{part} ==> epcoh ({epoch})\n"
        co=0
        for loss_name in self.losses_keys:
            loss_list = self.history[part][loss_name]

            loss = loss_list[-2]
            report += f"{loss_name} : {loss}"
            if((co+1)%3==0):
                report += "\n"
            else:
                report+=" - "
            co+=1  
                 
        print(report)
    def print_metrics_report(self,part,epoch,class_wise=False):
        
        report_temp = f"{part} avg metrics for epoch {epoch} :\n"
        report_class_wise_temp = ""
        avg_dice = 0
        avg_precision = 0
        avg_recall = 0
        for index , c in self.class_maps.items():
            dice = self.metric_history[part]["dice"][index][-1]
            precision = self.metric_history[part]["precision"][index][-1]
            recall = self.metric_history[part]["recall"][index][-1]
            avg_dice += dice
            avg_precision += precision
            avg_recall += recall
            if(class_wise):
                report_class_wise_temp += f"{c} => dice : {dice} p : {precision} , r : {recall}\n"
        
        avg_dice = avg_dice/(self.class_count-1)
        avg_precision = avg_precision/(self.class_count-1)
        avg_recall = avg_recall/(self.class_count-1)

        self.metric_avg_list[part]["dice"]+=[avg_dice]
        self.metric_avg_list[part]["precision"]+=[avg_precision]
        self.metric_avg_list[part]["recall"]+=[avg_recall]
        
        report_temp+=f"avg dice : {avg_dice} - avg precision : {avg_precision} - avg recall : {avg_recall}"

        if(class_wise):
            report_temp = report_temp + "\n" +report_class_wise_temp[:-1] #removing last \n
        print(report_temp)

# costume_nnunet_blocks

In [ ]:
def crop_dims():
    pass
class Conv(nn.Module):
    def __init__(self,in_c , out_c,p):
        super(Conv,self).__init__()
        self.layers =  nn.Sequential( 
            nn.Conv2d(
                in_channels = in_c , 
                out_channels = out_c ,
                kernel_size=3, 
                stride = 1 ,
                padding = p
            ),
            nn.InstanceNorm2d(out_c, eps=1e-5, affine=True),
            nn.LeakyReLU(negative_slope=1e-2, inplace=True)
        )
    def forward(self,x):
        return self.layers(x)
class DownsampleConv(nn.Module):
    def __init__(self,in_c , out_c):
        super(DownsampleConv,self).__init__()
        self.layers =  nn.Sequential( 
            nn.Conv2d(
                in_channels = in_c , 
                out_channels = out_c ,
                kernel_size=3, 
                stride = 2,
                padding=1
            ),
            nn.InstanceNorm2d(out_c, eps=1e-5, affine=True),
            nn.LeakyReLU(negative_slope=1e-2, inplace=True)
        )
    def forward(self,x):
        return self.layers(x)
class EncoderBlock(nn.Module):
    def __init__(self,in_c , out_c,p=1):
        super(EncoderBlock,self).__init__()
        self.layers = nn.Sequential(
            Conv(in_c = in_c , out_c = out_c , p=p),
            Conv(in_c = out_c , out_c=out_c ,p=p)
        )
        self.pool =  DownsampleConv(in_c = out_c , out_c=out_c )
    def forward(self,x):
        z = self.layers(x)
        return z , self.pool(z)

class DecoderBlock(nn.Module):
    def __init__(self,in_c ,out_c , f_int_scale,class_count,gate_c = None , attention=False,dsv=False):
        super(DecoderBlock,self).__init__()
        self.dsv=dsv
        self.conv1 = Conv(in_c=in_c , out_c=out_c,p=1)
        self.conv2 = Conv(in_c = out_c , out_c = out_c , p=1)
        self.upsampler = nn.ConvTranspose2d(
            in_channels = out_c , 
            out_channels = out_c//2 ,
            kernel_size=2 ,
            stride=2
        )

        if(self.dsv):
            self.dsv_block = nn.Conv2d(in_channels=out_c,out_channels=class_count,kernel_size=1)
        #in_c * 2 = gate_c
        if(attention):
            self.gate = AttentionGate(gate_in_c=gate_c ,f_int_scale=f_int_scale,skip_in_c=in_c//2)
        self.attention=attention
    def forward(self,x_in,x_skip,x_gate):
        if(self.attention):
            x_skip = self.gate(x_skip , x_gate)
        if(x_in.shape[2] != x_skip.shape[2] or x_in.shape[3] != x_skip.shape[3]):
            x_in = F.interpolate(
                x_in, 
                size=x_skip.shape[2:], 
                mode="bilinear", 
                align_corners=False
            )
        
        x = torch.cat([x_skip,x_in],dim=1)
        z = self.conv1(x)
        gate_z = self.conv2(z)
        upsampled_z = self.upsampler(gate_z)
        if(self.dsv):
            dsv_out = self.dsv_block(gate_z)
            
            if(self.attention):
               
                return upsampled_z , gate_z , dsv_out
            else:
                return upsampled_z , None, dsv_out 
        else:
            if(self.attention):
                return upsampled_z , gate_z , None
            else:
                return upsampled_z , None , None

class BottleNeck(nn.Module):
    def __init__(self,in_c , out_c,p,attention=False):
        super(BottleNeck,self).__init__()
        self.conv1 = Conv(in_c=in_c , out_c=out_c,p=p)
        self.conv2 = Conv(in_c = out_c , out_c = out_c , p=p)
        self.upsampler = nn.ConvTranspose2d(
            in_channels = out_c , 
            out_channels = out_c//2 ,
            kernel_size=2 ,
            stride=2
        )
        self.attention=attention
    def forward(self,x):
        z = self.conv1(x)
        gate_z = self.conv2(z)
        upsampled_z = self.upsampler(gate_z)
        if(self.attention):
            return upsampled_z , gate_z
        return upsampled_z , None

class AttentionGate(nn.Module):
    def __init__(self,gate_in_c,skip_in_c,f_int_scale=2,f_int=None,scaler="sigmoid"):
        super(AttentionGate,self).__init__()
        f_int = min(gate_in_c//f_int_scale,skip_in_c//f_int_scale) if f_int==None else f_int
        f_int = 1 if f_int == 0 else f_int
        self.conv_gate = nn.Conv2d(in_channels = gate_in_c , out_channels = f_int , 
                                   kernel_size = 1)
        self.conv_skip = nn.Conv2d(in_channels = skip_in_c , out_channels = f_int , 
                                   kernel_size = 1)
        self.relu = nn.ReLU(inplace=True)
        self.conv_shrink = nn.Conv2d(in_channels = f_int , out_channels = 1 ,
                                     kernel_size = 1)
        if(scaler =="sigmoid"):
            self.scaler = nn.Sigmoid()    
    def forward(self,x_skip,x_gate):
        x_gate_int = self.conv_gate(x_gate)
        x_skip_int = self.conv_skip(x_skip)
        # x_skip_int = crop_dims(x_gate_int,x_skip_int)
        if x_skip_int.shape[2:] != x_gate_int.shape[2:]:
            x_skip_int = F.interpolate(
                x_skip_int, 
                size=x_gate_int.shape[2:], 
                mode="bilinear", 
                align_corners=False
            )
        
        added_x = x_skip_int + x_gate_int
        relu_x = self.relu(added_x)
        shrinked_x = self.conv_shrink(relu_x)
        sig_x = self.scaler(shrinked_x)
        # padded_x = padd_dims(x_skip , sig_x)
        if sig_x.shape[2:] != x_skip.shape[2:]:
            padded_x = F.interpolate(
                sig_x, 
                size=x_skip.shape[2:], 
                mode="bilinear", 
                align_corners=False
            )

        return padded_x*x_skip

class Head(nn.Module):
    def __init__(self,in_c ,out_c ,class_count ,f_int_scale, 
        gate_c = None , attention=False):

        super(Head,self).__init__()
        self.conv1 = Conv(in_c=in_c , out_c=out_c,p=1)
        self.conv2 = Conv(in_c = out_c , out_c = out_c , p=1)
        self.conv1x1 = nn.Conv2d(
            in_channels = out_c , 
            out_channels = class_count ,
            kernel_size=1
        )
        
        if(attention):
            self.gate = AttentionGate(
                gate_in_c=gate_c , 
                f_int_scale=f_int_scale,
                skip_in_c=in_c//2
            )
        self.attention=attention
    def forward(self,x_in,x_skip,x_gate):
        if(self.attention):
            x_skip = self.gate(x_skip , x_gate)
        if(x_in.shape[2] != x_skip.shape[2] or x_in.shape[3] != x_skip.shape[3]):
            x_in = F.interpolate(
                x_in, 
                size=x_skip.shape[2:], 
                mode="bilinear", 
                align_corners=False
            )
            
        x = torch.cat([x_skip,x_in],dim=1)
        z = self.conv1(x)
        gate_z = self.conv2(z)
        
        class_feature_maps = self.conv1x1(gate_z) 
        
        return class_feature_maps

# nnunet

In [ ]:
class nnUnet(nn.Module):
    def __init__(self,args,encoder_channel_settings=None,decoder_channel_settings=None):
        super(nnUnet,self).__init__()
        
        in_c = args["in_c"]
        class_count = args["class_count"]
        attention = args["attention"]
        image_shape = args["image_shape"]
        base_channel = args["base_channel"]
        f_int_scale = args["f_int_scale"]
        max_channels = args["max_channels"]
        input_channels = args["input_channels"]
        self.deep_super_vision = args["deep_super_vision"]
        unet_depth = args["unet_depth"]
        h = image_shape[0]
        w = image_shape[1]
        
        co=0
        if(not unet_depth):
            while(w>4 and h>4):
                w/=2
                h/=2
                co+=1
            
        else:
            co = unet_depth
        print(f"number of layers : {co}")
        # create encoder settings 
        if(encoder_channel_settings is None):
            self.encoder_channel_settings = [base_channel]
            for i in range(co-1):
                new_c =min(self.encoder_channel_settings[i]*2,max_channels)
                self.encoder_channel_settings +=[new_c]
        else :
            self.encoder_channel_settings = encoder_channel_settings
        
        # create bottleneck settings
        self.bottle_neck_channel_setting = self.encoder_channel_settings[-1]*2
        # create decoder settings 
        if(decoder_channel_settings is  None):
            self.decoder_channel_settings =[]
            for i in range(co-1):
                self.decoder_channel_settings = [self.encoder_channel_settings[i]*2] +  self.decoder_channel_settings
        else :
            self.decoder_channel_settings = decoder_channel_settings

        
        # build encoder
        self.encoders = nn.ModuleList()
        for i in range(co):
            output_channels = self.encoder_channel_settings[i]
            self.encoders.append(EncoderBlock(in_c=input_channels,out_c=output_channels , p=1))
            input_channels = output_channels
        # build bottleneck

        self.bottle_neck = BottleNeck(in_c = output_channels ,out_c = self.bottle_neck_channel_setting , p=1,attention = attention)
        #build decoder
        input_channels = self.bottle_neck_channel_setting
        self.decoders = []
        for i in range(co-1):
            
            output_channels = self.decoder_channel_settings[i]

            self.decoders = [
                DecoderBlock(
                    in_c = input_channels , 
                    out_c=output_channels , 
                    gate_c = input_channels , 
                    attention = attention,
                    f_int_scale=f_int_scale,
                    dsv = self.deep_super_vision,
                    class_count=class_count
                )] + self.decoders
            
            input_channels = output_channels


        self.decoders = nn.ModuleList(self.decoders)
        self.attention = attention
        
        self.head = Head(
            in_c = input_channels , 
            out_c=input_channels//2 ,
            class_count = class_count,
            gate_c = input_channels , 
            attention = False,
            f_int_scale=f_int_scale
        )
        print("encoder settings : ", self.encoder_channel_settings)
        print("bottle-neck settings : ", self.bottle_neck_channel_setting)
        print("decoder settings : ", self.decoder_channel_settings)
        print("head settings : ",class_count)
    def forward(self,x):
        skips = []
        for encoder in self.encoders : 
            skip , out = encoder(x)
            # print(skip.shape)
            # print(out.shape)
            # print("======")
            skips += [skip]
            x = out
        x_in,gate_in = self.bottle_neck(x)
        
        outputs = []
        # print(len(self.decoders))
        for i in range(len(self.decoders) - 1, -1, -1):
            # print("2")
            decoder = self.decoders[i]
            skip = skips[i+1]
            x_out,gate_out,dsv_out = decoder(x_in,skip,gate_in)
            
            if(dsv_out!=None):
                outputs = [dsv_out] + outputs
            x_in=x_out
            gate_in=gate_out
        # print(x_in.shape)
        outputs = [self.head(
            x_in = x_in,
            x_skip = skips[0],
            x_gate = gate_in
        )] + outputs
        return outputs

# swin_encoder

In [ ]:
class SwinEncoder(nn.Module):
    def __init__(self,args):
        self.confs = {
            "swin_t" : [swin_t,224,96,4,Swin_T_Weights],
            "swin_s" : [swin_s,224,96,4,Swin_S_Weights],
            "swin_b" : [swin_b,224,128,4,Swin_B_Weights],
            "swin_v2_t" : [swin_v2_t,256,96,4,Swin_V2_T_Weights],
            "swin_v2_s" : [swin_v2_s,256,96,4,Swin_V2_S_Weights],
            "swin_v2_b" : [swin_v2_b,256,128,4,Swin_V2_B_Weights]
        }
        super(SwinEncoder,self).__init__()
        swin_head = args["swin_head"]
        swin_type = args["swin_type"]
        class_count = args["class_count"]
        abs_class_count = args["abs_class_count"]
        in_c = args["in_c"]
        deep_super_vision  = args["deep_super_vision"]
        swin_builder , base_img_size , emb_size , depth , weight_fn = self.confs[swin_type]
        self.backbone = swin_builder(weights=weight_fn.IMAGENET1K_V1)
        # print(self.backbone)
        if(in_c!=3):
            self.convert_base_channels(in_c)
        input_h,input_w =args["image_shape"] 
        if(swin_head=="costume"):
            self.head = CostumeHead(
                input_h=input_h,
                input_w=input_w,
                depth=depth,
                emb_size=emb_size,
                class_count = class_count,
                abs_class_count = abs_class_count,
                deep_super_vision=deep_super_vision

            )
    def convert_base_channels(self,in_c):
        old_conv = self.backbone.features[0][0]
        new_conv = nn.Conv2d(
            in_channels=in_c,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=(old_conv.bias is not None)
        )
        with torch.no_grad():
            new_conv.weight[:] = old_conv.weight.mean(dim=1,keepdim=True)
            if old_conv.bias is not None:
                new_conv.bias[:] = old_conv.bias
        self.backbone.features[0][0] = new_conv

    def forward(self,x):
        z = self.backbone.features(x)
        z = self.backbone.norm(z)
        z = self.backbone.permute(z)
        return self.head(z)

class SwinUperNet(nn.Module):
    def __init__(self,args):
        super(SwinUperNet,self).__init__()
        model_name = "openmmlab/upernet-swin-base"
        self.model = AutoModelForSemanticSegmentation.from_pretrained(model_name)
        self.conv1x1 = nn.Conv2d(
            in_channels=150,
            out_channels=args["class_count"],
            kernel_size=1
        )
    def forward(self,x):
        z = self.model(x).logits
        z = self.conv1x1(z)
        return [z]

# swin_blocks

In [ ]:
class CostumeBlock(nn.Module):
    def __init__(self,in_c,out_c):
        super(CostumeBlock,self).__init__()
        self.layer = nn.Sequential(
            nn.ConvTranspose2d(
                in_channels=in_c,
                out_channels=out_c,
                kernel_size=2,
                stride=2,
                bias=True
            ),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self,x):
        return self.layer(x)
class CostumeHead(nn.Module):
    def __init__(self,input_h,input_w,depth,emb_size,
                 class_count,abs_class_count,deep_super_vision):
        super(CostumeHead,self).__init__()
        self.layers = nn.ModuleList()
        self.dsv_layers = nn.ModuleList()

        self.input_w = input_w
        self.input_h = input_h
        self.deep_super_vision = deep_super_vision
        in_c = emb_size*(2**(depth-1))
        
        self.stage1 = CostumeBlock(in_c=in_c,out_c=in_c//2) # C//2 x H/16 x W/16
        in_c//=2

        self.stage2 = CostumeBlock(in_c=in_c,out_c=in_c//2) # C//4 x H/8 x W/8
        in_c//=2

        self.stage3 = CostumeBlock(in_c=in_c,out_c=in_c//2) # C//8 x H/4 x W/4
        in_c//=2
        
        self.stage4 = CostumeBlock(in_c=in_c,out_c=in_c//2) # C//16 x H/2 x W/2
        in_c//=2
        

        self.stage5 = CostumeBlock(in_c=in_c,out_c=in_c//2) # C//32 x H x W
        in_c//=2

        self.head = nn.Sequential(
            nn.Conv2d(
                in_channels=in_c, 
                out_channels=in_c//2,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(in_c//2),
            nn.LeakyReLU(),
            

            nn.Conv2d(
                in_channels=in_c//2,
                out_channels=in_c//2,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(in_c//2),
            nn.LeakyReLU(),

            nn.Conv2d(in_channels=in_c//2,out_channels=2,kernel_size=1)
        )
        
        
    def forward(self,x):
        z = self.stage1(x)

        z1 = self.stage2(z)
        z2 = self.stage3(z1)
        z3 = self.stage4(z2)

        z4 = self.stage5(z3)
        return [self.head(z4)]
        # torch.Size([8, 512, 24, 24]) 
        # torch.Size([8, 256, 48, 48]) 
        # torch.Size([8, 128, 96, 96]) 
        # torch.Size([8, 64, 192, 192]) 
        # torch.Size([8, 32, 384, 384])

# losses

In [ ]:
class MainLossFn(nn.Module):
    def __init__(self,args,eps = 1e-8):
        super(MainLossFn,self).__init__()
        class_count = args["class_count"]
        abs_class_count = args["abs_class_count"]
        self.loss_type = args["loss_type"]
        self.alpha = args["alpha"]
        self.beta = args["beta"]
        self.t_gamma = args["t_gamma"]
        self.f_gamma = args["f_gamma"]
        self.k = args["k"]
        self.loss_coefs = args["loss_coefs"]
        self.just_binary_trining = args["just_binary_trining"]
        remove_bg = args["remove_bg"]
        self.binary_type = args["binary_type"]
        entropy_fn = FocalCrossEntropy(
            f_gamma=self.f_gamma,
            eps=eps,
            f_alpha=args["f_alpha"]
        )
        # if(args["f_alpha"] is not None):
        #     w = torch.tensor(args["f_alpha"],dtype=torch.float32,device="cuda")
        #     self.entropy_fn = nn.CrossEntropyLoss(weight=w)
        # else :
        #     self.entropy_fn = nn.CrossEntropyLoss()
        
        self.eps = eps
        self.sum_dims = (0,2,3)
        if(self.loss_type=="dice loss"):
            print("loss is set to dice")
            multi_loss_fn = DiceLoss(self.eps,self.sum_dims)
        elif(self.loss_type=="tversky loss"):
            print("loss is set to tversky")
            multi_loss_fn = TverskyLoss(self.eps,self.sum_dims,self.alpha,
                                        self.beta,self.t_gamma)
            
        cldice_fn = CLDiceLoss(sum_dims=self.sum_dims,eps=self.eps,k=self.k) 
        
        bce_fn = nn.BCEWithLogitsLoss()
        self.mask_loss_fn = MultiClassLoss(
            class_count=class_count,
            multi_loss_fn=multi_loss_fn,
            entropy_fn=entropy_fn
        )
        
        self.abs_mask_loss_fn = MultiClassLoss(
            class_count=abs_class_count,
            multi_loss_fn=multi_loss_fn,
            entropy_fn=entropy_fn
        )
        
        self.binary_mask_loss_fn =BinaryClassLoss(
            cldice_fn=cldice_fn,
            dice_fn=TverskyLoss(
                self.eps,
                self.sum_dims,
                self.alpha,
                self.beta,
                self.t_gamma
            ),
            remove_bg = remove_bg
            # dice_fn=DiceLoss(self.eps,self.sum_dims)
        )
        
        self.label_loss_fn = SideLoss(
            entropy_fn=bce_fn
        )
        
        
        
    def forward(self,preds , ground_truths ):
        if(self.just_binary_trining):
            gt_common_mask , gt_rare_mask = ground_truths
            if(self.binary_type=="common"):
                gt_binary_mask = gt_common_mask
            else:
                gt_binary_mask = gt_rare_mask

            pred_binary_mask = preds[0]
            binary_mask_loss , dice_loss  , bce_loss = self.binary_mask_loss_fn(
                pred_binary_mask = pred_binary_mask,
                gt_binary_mask = gt_binary_mask
            )
            total_loss = (
                binary_mask_loss 
            )
            loss_dict = {
                "binary loss" : binary_mask_loss,
                # "bianry cldice loss ": cldice_loss,
                "binary dice loss":dice_loss,
                "binary BCE loss":bce_loss
            }
        else:
            gt_side_label,gt_binary_mask,gt_abs_mask,gt_mask = ground_truths
            pred_side_label ,pred_abs_mask, pred_mask = preds
        
            label_loss = self.label_loss_fn(
                pred_label = pred_side_label,
                gt_label=gt_side_label
            )
            

            abs_mask_loss = self.abs_mask_loss_fn(
                pred_mask = pred_abs_mask,
                gt_mask = gt_abs_mask
            )

            mask_loss = self.mask_loss_fn(
                pred_mask = pred_mask,
                gt_mask = gt_mask
            )
            loss_dict = {
                "label loss" : label_loss,
                f"{self.loss_type}_main" : mask_loss,
                f"{self.loss_type}_abs" : abs_mask_loss,
            }
        
        return total_loss , loss_dict
    
class MultiClassLoss(nn.Module):
    def __init__(self,class_count,multi_loss_fn,entropy_fn):
        super(MultiClassLoss,self).__init__()
        self.class_count = class_count
        self.softmax = nn.Softmax(dim=1)
        self.multi_loss_fn = multi_loss_fn
        self.entropy_fn = entropy_fn
    def forward(self,pred_mask,gt_mask):

        onehot_mask = F.one_hot(gt_mask, num_classes=self.class_count)
        onehot_mask = onehot_mask.permute(0, 3, 1, 2).float()  
        prob = self.softmax(pred_mask)

        forground_prob = prob[:,1:]
        forground_onehot_mask = onehot_mask[:,1:]

        second_loss = self.multi_loss_fn(
            pred_probs = forground_prob,
            gt = forground_onehot_mask
        )

        en_loss = self.entropy_fn(prob,onehot_mask)

        return second_loss + en_loss
    
class SideLoss(nn.Module):
    def __init__(self,entropy_fn):
        super(SideLoss,self).__init__()    
        self.entropy_fn = entropy_fn
    def forward(self,pred_label,gt_label):
        en_loss = self.entropy_fn(
            pred_label.reshape(-1),
            gt_label.reshape(-1).float()
        )
        return en_loss

class BinaryClassLoss(nn.Module):
    def __init__(self,cldice_fn,dice_fn,remove_bg):
        super(BinaryClassLoss,self).__init__()
        self.cldice_fn = cldice_fn
        self.dice_fn = dice_fn
        self.coef =0.4
        self.softmax =nn.Softmax(dim=1)
        # self.cross_etropy = nn.CrossEntropyLoss()
        self.cross_etropy = FocalCrossEntropy(
            f_gamma=1.5,
            eps=1e-6,
            f_alpha=[1.0,3.0]
        )
        self.remove_bg = remove_bg
    def forward(self,pred_binary_mask,gt_binary_mask):
        # bce_loss = self.cross_etropy(
        #     pred_binary_mask,
        #     gt_binary_mask.squeeze(1)
        # )
        onehot_mask = F.one_hot(gt_binary_mask.squeeze(1), num_classes=2) # B,H,W,2
        onehot_gt_binary_mask = onehot_mask.permute(0, 3, 1, 2).float()  # B,2,H,W
        prob = self.softmax(pred_binary_mask)

        bce_loss = self.cross_etropy(
            prob = prob,
            onehot_mask = onehot_gt_binary_mask
        )
        if(self.remove_bg):
            # cldice_loss = self.cldice_fn(
            #     binary_pred = prob[:,1:2,...],
            #     binary_gt = gt_binary_mask.float()
            # )
            dice_loss = self.dice_fn(
                pred_probs = prob[:,1:2,...],
                gt = gt_binary_mask
            )
        else :     
            # cldice_loss_fg = self.cldice_fn(
            #     binary_pred = prob[:,1:2,...],
            #     binary_gt = onehot_gt_binary_mask[:,1:2,...]
            # )
            # cldice_loss_bg = self.cldice_fn(
            #     binary_pred = prob[:,0:1,...],
            #     binary_gt = onehot_gt_binary_mask[:,0:1,...]
            # )
            # cldice_loss = 0.5*(cldice_loss_fg+cldice_loss_bg)

            dice_loss = self.dice_fn(
                pred_probs = prob,
                gt = onehot_gt_binary_mask
            )
        loss = (
            bce_loss + 
            dice_loss  
            
        )
        ##(self.coef)*cldice_loss
        return loss, dice_loss , bce_loss
    
class FocalCrossEntropy(nn.Module):
    def __init__(self,f_gamma,eps,f_alpha=None):
        super(FocalCrossEntropy,self).__init__()
        self.f_gamma = f_gamma
        if(f_alpha is not None):
            device = "cuda" if torch.cuda.is_available() else "cpu"
            self.f_alpha = torch.tensor(f_alpha).to(device)
        else:
            self.f_alpha = f_alpha
        self.eps = eps
    def forward(self,prob,onehot_mask):
        # prob : (B,C,H,W)
        # onehot_mask : (B,C,H,W)
        # gt_mask = (B,H,W)

        p = (prob*onehot_mask).sum(dim=1) # (B,H,W)
        pt = torch.clamp(p,self.eps,1-self.eps)
        focal_weights = (1-pt)**self.f_gamma
        focal_loss = focal_weights*(torch.log(pt))
        if(self.f_alpha is not None):
            alpha_b = self.f_alpha.view(1, -1, 1, 1).type_as(prob)
            class_w = (alpha_b*onehot_mask).sum(dim=1)
        else :
            class_w = 1.0
        return -(class_w*focal_loss).mean()

class CLDiceLoss(nn.Module):
    def __init__(self,eps,sum_dims,k=40):
        super(CLDiceLoss,self).__init__()
        self.k=k
        self.eps = eps
        self.sum_dims = sum_dims

    def forward(self,binary_pred , binary_gt):
        # print(binary_pred.shape,binary_gt.shape,gt_skel.shape)
        pred_skel = soft_skeletonize(binary_pred,k=self.k)
        gt_skel = soft_skeletonize(binary_gt,k=self.k)
        
        num_prec = (pred_skel * binary_gt).sum(dim=self.sum_dims) + self.eps
        den_prec = pred_skel.sum(dim=self.sum_dims) + self.eps
        t_prec   = num_prec / den_prec

        num_rec = (gt_skel * binary_pred).sum(dim=self.sum_dims) + self.eps
        den_rec = gt_skel.sum(dim=self.sum_dims) + self.eps
        t_rec   = num_rec / den_rec

        cldice = 2 * ( (t_prec * t_rec) / (t_prec + t_rec + self.eps) )
        cldice_loss = 1 - cldice.mean()
        return cldice_loss
class DiceLoss(nn.Module):
    def __init__(self,eps,sum_dims):
        super(DiceLoss,self).__init__()
        self.eps = eps
        self.sum_dims = sum_dims
    def forward(self,pred_probs,gt):
        tp = (gt * pred_probs).sum(dim=self.sum_dims)
        fp = ((1-gt)*pred_probs).sum(dim=self.sum_dims)
        fn = ((1-pred_probs)*gt).sum(dim=self.sum_dims)
        per_class_dice_score = (2*tp +self.eps)/(2*tp + fp + fn + self.eps)
        dice_loss = 1 - per_class_dice_score.mean()
        return dice_loss

class TverskyLoss(nn.Module):
    def __init__(self,eps,sum_dims,alpha=0.3,beta=0.7,gamma=1.33):
        super(TverskyLoss,self).__init__()
        self.eps = eps
        self.sum_dims = sum_dims
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
    def forward(self,pred_probs,gt):
        tp = (gt * pred_probs).sum(dim=self.sum_dims)
        fp = ((1-gt)*pred_probs).sum(dim=self.sum_dims)
        fn = ((1-pred_probs)*gt).sum(dim=self.sum_dims)
        t_index = (tp + self.eps) / (tp + self.alpha*fp + self.beta*fn + self.eps) 

        t_index = t_index.mean()
        
        return (1 - t_index)**self.gamma

# trainer

In [ ]:
def model_sanity_check(model,loss,total_norm):
    if random.random() < 0.05:
        with torch.no_grad():
            print("--- Total Norm ---")
            print(loss,total_norm)
            if(random.random() < 0.005):
                print("\n--- Gradient norms ---")
                for name, param in model.named_parameters():
                    if param.grad is not None:
                        grad_norm = param.grad.data.norm().item()
                        data = torch.norm(param).item()
                        print(f"{name:30s}: {grad_norm:.6f} - {data:.6f}")
            print("----------------------\n")

    
def train_fn(model,img,ground_truths,optimizer,loss_fn,scaler,args,device,loss_weights=[1],use_amp=False):
    optimizer.zero_grad()
    loss_dict={}

    if(use_amp):
        with torch.autocast(device_type=args["device"],dtype=torch.float16,enabled=use_amp):
            preds =  model(img)
            loss , loss_dict = loss_fn(preds , ground_truths)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)
        
        model_sanity_check(model,loss,total_norm)
        
        scaler.step(optimizer)
        scaler.update()
    
    else:

        with torch.autocast(device_type=args["device"],dtype=torch.float16,enabled=use_amp):
            preds =  model(img)
            loss , loss_dict = loss_fn(preds , ground_truths)

        loss.backward()
        total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)

        model_sanity_check(model,loss,total_norm)

        optimizer.step()

    loss = loss.detach().cpu().item()
    for loss_name in loss_dict:
        loss_dict[loss_name] = loss_dict[loss_name].detach().cpu().item()
    
    loss_dict["total loss"] = loss
    pred_mask = preds[-1].detach()
    return loss_dict , pred_mask 
    
def trainer(args,recorder,model,optimizer,loss_fn,train_loader,valid_loader,lr_sch=None,loss_weights=[1]):
    device = args["device"]
    epcohs = args["epcohs"]
    class_count = args["class_count"]
    full_report_cycle = args["full_report_cycle"]
    use_amp = args["use_amp"]
    just_binary_trining = args["just_binary_trining"]
    binary_type = args["binary_type"]
    scaler = torch.amp.GradScaler(device = device,init_scale=2**8)
    best_val_dice = float("-inf")
    
    best_model = copy.deepcopy(model)
    best_ep = 0
    for ep in tqdm(range(epcohs)):
        total_TP =  torch.zeros(class_count)
        total_FP = torch.zeros(class_count)
        total_FN = torch.zeros(class_count)

        model.train()
        class_wise_report = False
        for img,gt_common_mask,gt_rare_mask,gt_mask in tqdm(train_loader) : 
            # gt_mask = gt_mask.long()
            img = img.to(device)
            if(just_binary_trining):
                gt_common_mask = gt_common_mask.to(device)
                gt_rare_mask = gt_rare_mask.to(device)

                ground_truths = [
                    gt_common_mask,
                    gt_rare_mask
                ]
            else:
                gt_mask = gt_mask.to(device)
                ground_truths = [
                    gt_mask
                ]

            loss_dict , pred_mask = train_fn(
                model = model,
                img = img,
                ground_truths = ground_truths,
                optimizer = optimizer,
                loss_fn = loss_fn,
                scaler = scaler,
                args = args,
                device = device,
                loss_weights = loss_weights,
                use_amp = use_amp
            )
            if(just_binary_trining):
                if(binary_type=="common"):
                    ground_truth_mask = gt_common_mask.squeeze(1)
                else:
                    ground_truth_mask = gt_rare_mask.squeeze(1)
            else:
                ground_truth_mask = gt_mask
            # labels_hist[0]+=[(F.sigmoid(pred_labels.detach().cpu().reshape(-1))>=0.5).tolist()]
            # labels_hist[1]+=[gt_side_label.cpu().reshape(-1).tolist()]
            # TP , _ , FP , FN = TP_TN_FP_FN(
            #     pred_mask,
            #     gt_mask,
            #     process_preds=True
            # )
            TP , _ , FP , FN = TP_TN_FP_FN(
                pred_mask,
                ground_truth_mask,
                process_preds=True
            )
            total_TP += TP
            total_FP += FP
            total_FN += FN
            
            recorder.add_losses("train",loss_dict)
            

        current_lr = [group['lr'] for group in optimizer.param_groups][0]
        print(f"current lr : {current_lr:.4}")
        # print("train acc",accuracy_score(labels_hist[1],labels_hist[0]))
        if(lr_sch is not None):
            lr_sch.step()
        

        dice_score = (2 * total_TP + 1e-8) / (2 * total_TP + total_FP + total_FN + 1e-8)

        precision = total_TP /(total_FP + total_TP + 1e-8) 
        recall = total_TP /(total_FN + total_TP + 1e-8) 
        
        recorder.add_metrics(
            dice_score.tolist(),
            precision.tolist(),
            recall.tolist(),
            part = "train"
        )
        
        recorder.print_loss_report("train",ep)
        recorder.print_metrics_report("train",ep,class_wise=False)
        print("<=>"*20)
        
        if((ep+1)%full_report_cycle==0):
            class_wise_report=True
            
        val_dice = evaluation(
            recorder=recorder,
            model=model,
            loss_fn=loss_fn,
            valid_loader=valid_loader,
            class_wise_report=class_wise_report,
            class_count = class_count,
            epoch=ep,
            device=device,
            use_amp = use_amp,
            just_binary_trining = just_binary_trining,
            binary_type=binary_type
        )
        if(val_dice>best_val_dice):
            print(f"New Best! : dice = {val_dice}")
            best_model = copy.deepcopy(model)
            best_val_dice = val_dice
            best_ep = ep + 1
    print(f"best result at epoch {best_ep} with dice {best_val_dice}")
    return best_model
@torch.no_grad()
def evaluation(recorder,model,loss_fn,valid_loader,class_count,binary_type,
               class_wise_report=False,epoch=None,device="cuda",
               use_amp=True,just_binary_trining=False):
    model.eval()
    total_TP = torch.zeros(class_count)
    total_FP = torch.zeros(class_count)
    total_FN = torch.zeros(class_count)

    labels_hist = [[],[]]
    
    for img,gt_common_mask,gt_rare_mask,gt_mask in valid_loader:
        img = img.to(device)
        if(just_binary_trining):
                gt_common_mask = gt_common_mask.to(device)
                gt_rare_mask = gt_rare_mask.to(device)

                ground_truths = [
                    gt_common_mask,
                    gt_rare_mask,
                ]
        else:
            gt_mask = gt_mask.to(device)
            ground_truths = [
                gt_mask
            ]
        with torch.autocast(device_type=device,dtype=torch.float16,enabled=use_amp):
            preds  = model(img)
            loss , loss_dict = loss_fn(preds , ground_truths)

            loss = loss.detach().cpu().item()
            for loss_name in loss_dict:
                loss_dict[loss_name] = loss_dict[loss_name].detach().cpu().item()
        
            loss_dict["total loss"] = loss
        if(just_binary_trining):
            if(binary_type=="common"):
                ground_truth_mask = gt_common_mask.squeeze(1)
            else:
                ground_truth_mask = gt_rare_mask.squeeze(1)
        else:
            ground_truth_mask = gt_mask

        pred_mask = preds[-1]
        # labels_hist[0]+=[(F.sigmoid(pred_labels.detach().cpu().reshape(-1))>=0.5).tolist()]
        # labels_hist[1]+=[gt_side_label.cpu().reshape(-1).tolist()]
        
        # TP , _ , FP , FN = TP_TN_FP_FN(pred_mask,gt_mask,process_preds=True)

        TP , _ , FP , FN  = TP_TN_FP_FN(
            pred_mask,
            ground_truth_mask,
            process_preds=True
        )
        total_TP += TP
        total_FP += FP
        total_FN += FN
        

        recorder.add_losses("valid",loss_dict)

    # print("test acc",accuracy_score(labels_hist[1],labels_hist[0]))
    dice_score = (2 * total_TP + 1e-8) / (2 * total_TP + total_FP + total_FN + 1e-8)
    precision = total_TP /(total_FP + total_TP + 1e-8) 
    recall = total_TP /(total_FN + total_TP + 1e-8) 

    recorder.add_metrics(
        dice_score.tolist(),
        precision.tolist(),
        recall.tolist(),
        part = "valid"
    )
    recorder.print_loss_report("valid",epoch)
    recorder.print_metrics_report("valid",epoch,class_wise=class_wise_report)
    print("-"*60)
    return dice_score[1:].mean().item()

# temp_script

In [ ]:


# In[2]:


args = {
    "base_path" : "./dataset/syntax",
    "in_c" : 3,
    "base_channel" :32,
    "image_shape" : (448,448),
    "abs_class_count":17,
    "attention" : True,
    "k":40,
    "batch_size" : 10,
    "num_workers" : 10,
    "device" : "cuda" if torch.cuda.is_available() else "cpu",
    "lr" : 1e-2,
    "momentum" : 0.99,
    "weight_decay" : 0.001,
    "epcohs":30,
    "f_int_scale" : 2,
    "full_report_cycle" : 10,
    "max_channels":512,
    "unet_depth":6,
    "loss_type":"tversky loss",
    "alpha":0.55,
    "beta":0.45,
    "t_gamma":2.0,
    "f_gamma":2.0,
    "resize_binary":[True,(224,224)],
    "loss_coefs":{"CE":1.0,"Second":1.0},
    "swin_head" : "costume",
    "swin_type":"swin_v2_b",
    "output_base_path" : "./outputs",
    "name" : "common-amp32-weighted",
    "deep_super_vision" : False,
    "just_binary_trining":True,
    "use_sch":True,
    "use_amp":True,
    "f_alpha":None,
    "remove_bg":True,
    "binary_type":"common"
}
args["class_count"] = 2 if args["just_binary_trining"] else 26
if args["just_binary_trining"]:
    class_map = {
        1:"fg"
    }

else:
    class_map = {
        1: '1',2: '2', 3: '3',4: '4',
        5: '5',6: '6',7: '7',8: '8',
        9: '9',10: '9a',11: '10',12: '10a',
        13: '11',14: '12',15: '12a',16: '13',
        17: '14',18: '14a',19: '15',20: '16',
        21: '16a',22: '16b',23: '16c',
        24: '12b',25: '14b'
    }

rare = {
    21: '16a',22: '16b',23: '16c',
    24: '12b',25: '14b',18: '14a',
    10: '9a',12: '10a',15: '12a',11: '10'
}

common ={
    1: '1',2: '2', 3: '3',4: '4',
    5: '5',6: '6',7: '7',8: '8',
    9: '9',13: '11',14: '12',19: '15',
    20: '16',16: '13',17: '14'
}

train_class_counts = [
    1000,374,375,369,303,525,525,
    340,310,198,70,21,1,320,61,
    129,305,107,49,38,232,43,48,31,63,127
]
train_pixel_counts = [
    253576361,664435,686727,661957,
    480566,591829,816901,685677,570436,
    470633,124025,23866,1079,507754,151219,
    336857,597880,241117,98167,66890,322098,
    49426,63543,36457,164558,153542
]
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
# losses_keys = ["total loss","FCE loss",args["loss_tgrad_normype"]]

losses_keys = [
    "total loss",
    "binary loss",
    # "bianry cldice loss ",
    "binary dice loss",
    "binary BCE loss"
    # f"{args["loss_type"]}_abs",
    # f"{args["loss_type"]}_main",
]
out_counts = 5 if args["deep_super_vision"] else 1
loss_weights = [1/(2**i) for i in range(out_counts)]
loss_weights


# In[3]:


def class_weighting(method,class_counts,**kwargs):
    if(kwargs["use_pixel_counts"]):
        print("using pixel counts")
        with open("./data/train_pixel_counts.json","r") as f:
            train_class_counts = json.load(f)
        counts = [0]*(len(train_class_counts))
        for k,v in train_class_counts.items():
            counts[int(k)] = int(v)
        counts = np.array(counts,dtype=np.float64)
    else :
        print("using class counts")
        counts = np.array(class_counts,dtype=np.float64)

    if(method=="median"):
        print("median weights being used")
        median_count = np.median(counts)
        weights = median_count/np.array(counts)

    elif(method=="log"):
        print("log weights being used")
        total = np.sum(counts)
        weights = np.log(total/np.array(counts))
        weights = (weights / weights.mean())
        weights[0]=0.1
    elif(method=="beta"):
        print("beta weights being used")
        b = kwargs["b"]
        weights = (1-b)/(1-np.power(b,counts))
        weights = weights / weights.sum()
        weights[12] = 0.25
    else:
        print("no class weights being used")
        return None
    return weights.tolist()
args["f_alpha"] = class_weighting(method="none",class_counts=train_class_counts,b=0.999999,use_pixel_counts=False)
args["f_alpha"]


# In[4]:


train_transforms = A.Compose([
    # A.RandomCrop(args["image_shape"][0],args["image_shape"][1]),
    A.Resize(*args["image_shape"]),
    A.OneOf([
        A.ElasticTransform(
            alpha=120, 
            sigma=120 * 0.05, 
            p=1.0
        ),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0),
        A.OpticalDistortion(distort_limit=0.2, p=1.0),
    ], p=0.7),


    A.Affine(
        scale=(0.8, 1.2),             
        translate_percent=(-0.1, 0.1), 
        rotate=(-30, 30),         
        shear=(-10, 10),      


        fill=0,           
        fill_mask=0,                 
        border_mode=cv2.BORDER_CONSTANT, 

        fit_output=False,  
        p=0.7
    ),

    # A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),

    # A.RandomBrightnessContrast(
    #     brightness_limit=0.2, 
    #     contrast_limit=0.2, 
    #     p=0.5
    # ),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    # A.Lambda(image=morph_binary_mask, p=1),

    # A.Lambda(image=normalize_xca),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
        max_pixel_value=255.0
    ),
    A.ToTensorV2()

],additional_targets={'common_mask': 'mask', 'rare_mask': 'mask'})

test_transforms = A.Compose([
    # A.RandomCrop(args["image_shape"][0],args["image_shape"][1]),
    A.Resize(*args["image_shape"]),
    # A.Lambda(image=normalize_xca),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
        max_pixel_value=255.0
    ),
    A.ToTensorV2()
],additional_targets={'common_mask': 'mask', 'rare_mask': 'mask'})
# train_preprocess = v2.Compose([
    # WhiteTopHat(kernel_size=(50,50)),
    # CLAHE()
    # DoubleClahe(kernel_size=(3,3),tile_grid=(10,10))
    # laplacian_pyramid_enhance
# ])
train_preprocess = None


# In[5]:


train_images,sampler_weights = read_images(
    base_path = args["base_path"],
    preprocessor = train_preprocess,
    part = "train",
    train_class_counts=np.array(train_pixel_counts),
    in_c = args["in_c"],
    resize_binary = args["resize_binary"],
    k = args["k"],
    rare=rare
)
valid_images = read_images(
    base_path = args["base_path"],
    preprocessor = train_preprocess,
    part = "val",
    train_class_counts=None,
    in_c=args["in_c"],
    resize_binary = args["resize_binary"],
    k = args["k"],
    rare=rare
)
# print(sampler_weights)
train_ds = UnetDataset(
    transform = train_transforms,
    data = train_images,
    base_size=args["image_shape"]
)
valid_ds = UnetDataset(
    transform = test_transforms,
    data = valid_images,
    base_size=args["image_shape"]
)

train_loader = make_dataloader(train_ds,args,valid=False,sampler_weights=None)
valid_loader = make_dataloader(valid_ds,args,valid=True,sampler_weights=None)


# In[6]:


colors = np.array([
    (242,  24,  24),   # Red
    (242,  77,  24),   # Red-Orange
    (242, 129,  24),   # Orange
    (242, 181,  24),   # Yellow-Orange
    ( 24, 242, 216),   # Cyan
    (242, 234,  24),   # Yellow
    (146,  24, 242),   # Purple
    (199, 242,  24),   # Yellow-Green
    (146, 242,  24),   # Lime
    ( 94, 242,  24),   # Green
    (242,  24, 181),   # Fuchsia
    ( 42, 242,  24),   # Green (brighter)
    ( 94,  24, 242),   # Violet
    ( 24, 242,  59),   # Spring Green
    (242,  24, 129),   # Pink
    ( 24, 242, 111),   # Aquamarine
    ( 24, 242, 164),   # Turquoise
    ( 24, 164, 242),   # Azure
    (199,  24, 242),   # Magenta
    ( 24, 216, 242),   # Sky Blue
    ( 24, 111, 242),   # Blue
    (242,  24, 234),   # Hot Pink
    ( 24,  59, 242),   # Royal Blue
    ( 42,  24, 242),   # Indigo
    (242,  24,  77),   # Rose
], dtype=np.uint8)

for img,common_mask,rare_mask,mask in train_loader:
    print(img.shape)
    print(common_mask.shape)
    print(rare_mask.shape)
    print(mask.shape)
    ### binary check 
    index=2
    print(np.unique(common_mask[index].numpy()))
    print(np.unique(rare_mask[index].numpy()))
    print(np.unique(mask[index].numpy()))
    ### abs check 
    img = denorm(img[index],mean=IMAGENET_MEAN,std=IMAGENET_STD)
    plt.figure(figsize=(10,10))

    plt.subplot(2,2,1)
    plt.imshow(img)
    plt.subplot(2,2,2)
    colored_25 = draw_mask(image=img,mask=mask[index].numpy(),colors=colors)
    plt.imshow(colored_25)
    plt.subplot(2,2,3)
    plt.imshow(common_mask[index][0].numpy(),cmap="gray")
    plt.subplot(2,2,4)
    plt.imshow(rare_mask[index][0].numpy(),cmap="gray")
    break


# In[7]:


class UnetP(torch.nn.Module):
    def __init__(self):
        super(UnetP,self).__init__()
        self.model = smp.UnetPlusPlus(
            encoder_name="resnet50",  
            encoder_weights="imagenet",  
            in_channels=3,               
            classes=2
        )
    def forward(self,x):
        return [self.model(x)]


# In[ ]:


# model = SwinEncoder(args).to(args["device"])
model = UnetP().to(args["device"])
loss_fn = MainLossFn(args)

# optimizer = torch.optim.Adam(model.parameters(), lr=args["lr"])
optimizer = torch.optim.SGD(
    model.parameters(),
    momentum=args["momentum"],
    lr=args["lr"],
    nesterov=True,
    weight_decay=args["weight_decay"]
)
# optimizer = torch.optim.AdamW(
#     model.parameters(), 
#     lr=args["lr"], 
#     betas=(0.9, 0.999), 
#     eps=1e-08, 
#     weight_decay=args["weight_decay"]
# )
if(args["use_sch"]):
    lr_sch = PolynomialLR(optimizer=optimizer,total_iters=args["epcohs"],power=0.9)
else:
    lr_sch = None

recorder = HistoryRecorder(losses_keys=losses_keys,class_maps =class_map,class_count=args["class_count"])

best_model =trainer(
    args=args,
    recorder = recorder,
    model = model,
    optimizer = optimizer,
    loss_fn = loss_fn,
    train_loader = train_loader,
    valid_loader = valid_loader,
    loss_weights=loss_weights,
    lr_sch = lr_sch
)
# ls = torch.rand(2,3,448,448).to("cuda")


# In[ ]:


save_full_report(
    recorder= recorder , 
    output_base_path=args["output_base_path"],
    model=best_model,
    valid_loader=valid_loader,
    args=args,
    class_map=class_map,
    name=args["name"],
    mean=IMAGENET_MEAN,
    std=IMAGENET_STD,
    just_binary_trining = args["just_binary_trining"],
    use_amp = args["use_amp"],
    binary_type = args["binary_type"]
)


# In[ ]:





# In[ ]:





# In[ ]:





# In[ ]: